# Assignment 1 — Build a Custom Missing-Value Imputer
**Course:** Feature Engineering & MLOps · Unit 1, Session 4 follow-up (Missing Values)

**Dataset:** `student_performance_raw.csv` (PrepEdge coaching-institute dataset)

This notebook implements a scikit-learn–compatible `CustomImputer` class, applies it to the
PrepEdge dataset with strict train/test discipline, verifies correctness against `SimpleImputer`,
and answers the reflection questions.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted
import pandas.api.types as ptypes

pd.set_option("display.max_columns", None)
np.random.seed(42)


## 2. Section 4.1 — Design the `CustomImputer` class

The class follows scikit-learn's estimator API:

- Inherits from `BaseEstimator` and `TransformerMixin` (giving us `.fit_transform()` for free, and
  `get_params`/`set_params` compatibility with pipelines/grid search).
- `fit(X, y=None)` learns and **stores** a fill value per column (never recomputes in `transform`).
- `transform(X)` returns a **copy** of `X` with missing values filled, plus optional
  `<column>_was_missing` indicator columns.
- Column type is **auto-detected** with `pandas.api.types` rather than requiring the user to specify it.
- Calling `transform()` before `fit()` raises a clear error via `check_is_fitted`.
- A bonus `column_overrides` parameter lets individual columns use a different strategy than the
  dataset-wide default (Section 7).


In [2]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """A scikit-learn-compatible imputer for missing values.

    Follows the same fit/transform contract as sklearn's built-in transformers
    (e.g. SimpleImputer). Numeric columns are filled with the mean or median;
    categorical (and text) columns are filled with the mode (most frequent value).
    Optionally adds a binary "<column>_was_missing" indicator column for every
    column that had missing values in the TRAINING data, which is useful for
    flagging MNAR/MAR missingness patterns rather than pretending they don't exist.

    Parameters
    ----------
    numeric_strategy : {'mean', 'median'}, default='median'
        Statistic used to fill missing values in numeric columns, unless
        overridden per-column via `column_overrides`.
    categorical_strategy : {'most_frequent'}, default='most_frequent'
        Strategy used to fill missing values in categorical/text columns.
        (Only 'most_frequent' is currently supported for categoricals.)
    add_missing_indicator : bool, default=True
        If True, add a binary "<column>_was_missing" column for every column
        that contained at least one missing value in the training data.
    column_overrides : dict, optional
        Bonus (Section 7): a mapping of {column_name: strategy} that overrides
        the dataset-wide default strategy for specific columns, e.g.
        {'weekly_study_hours': 'mean', 'income_bracket': 'most_frequent'}.

    Attributes
    ----------
    fill_values_ : dict
        Fitted fill value for every column, learned from the training data.
    columns_with_missing_ : list
        Columns that had at least one missing value during fit (used to decide
        which "_was_missing" indicator columns to create).
    column_types_ : dict
        {'numeric' or 'categorical'} per column, decided during fit.
    feature_names_in_ : list
        Column names seen during fit, in order.
    """

    def __init__(self, numeric_strategy="median", categorical_strategy="most_frequent",
                 add_missing_indicator=True, column_overrides=None):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _is_numeric(self, series):
        """Return True if a pandas Series should be treated as numeric."""
        return ptypes.is_numeric_dtype(series)

    def _compute_fill_value(self, series, strategy):
        """Compute a single fill value for one column given a strategy name."""
        if strategy == "mean":
            return series.mean()
        elif strategy == "median":
            return series.median()
        elif strategy == "most_frequent":
            mode = series.mode(dropna=True)
            # If every value is missing there is no mode; fall back to a
            # placeholder so transform() never crashes on an all-NaN column.
            return mode.iloc[0] if len(mode) > 0 else "missing"
        else:
            raise ValueError(f"Unknown strategy: {strategy!r}")

    def fit(self, X, y=None):
        """Learn the fill value for every column in X.

        Parameters
        ----------
        X : pandas.DataFrame
            Training data. Statistics are computed from this data only.
        y : ignored
            Present for scikit-learn API compatibility.

        Returns
        -------
        self : CustomImputer
            The fitted transformer.
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        overrides = self.column_overrides or {}
        self.feature_names_in_ = list(X.columns)
        self.column_types_ = {}
        self.fill_values_ = {}
        self.columns_with_missing_ = [c for c in X.columns if X[c].isna().any()]

        for col in X.columns:
            series = X[col]
            is_numeric = self._is_numeric(series)
            self.column_types_[col] = "numeric" if is_numeric else "categorical"

            # Decide which strategy applies to this column: per-column override
            # wins, otherwise fall back to the dataset-wide default for its type.
            if col in overrides:
                strategy = overrides[col]
            elif is_numeric:
                strategy = self.numeric_strategy
            else:
                strategy = self.categorical_strategy

            self.fill_values_[col] = self._compute_fill_value(series, strategy)

        return self

    def transform(self, X):
        """Fill missing values in X using statistics learned during fit().

        Parameters
        ----------
        X : pandas.DataFrame
            Data to transform (training, test, or new data). Must contain the
            same columns seen during fit.

        Returns
        -------
        pandas.DataFrame
            A COPY of X with missing values filled and, if
            `add_missing_indicator=True`, one extra "<column>_was_missing"
            binary column per column that had missing values during fit.
        """
        check_is_fitted(self, attributes=["fill_values_"])

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        X = X.copy()  # never mutate the caller's data

        # Add "_was_missing" indicators BEFORE filling, so they reflect the
        # missingness pattern of the data being transformed (train or test).
        if self.add_missing_indicator:
            for col in self.columns_with_missing_:
                if col in X.columns:
                    X[f"{col}_was_missing"] = X[col].isna().astype(int)

        for col, fill_value in self.fill_values_.items():
            if col in X.columns:
                X[col] = X[col].fillna(fill_value)

        return X


### Quick guard-rail check

Calling `transform()` before `fit()` should raise a clear, informative error.

In [3]:
try:
    CustomImputer().transform(pd.DataFrame({"a": [1, np.nan]}))
except Exception as e:
    print(f"Raised as expected: {type(e).__name__}: {e}")


Raised as expected: NotFittedError: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 3. Section 4.2 — Apply it to the dataset

### 3.1 Load the data and inspect missingness

In [4]:
df = pd.read_csv("student_performance_raw.csv")
print(df.shape)
df.head()


(600, 17)


,student_id,city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score
0,1447,Patna,2,JEE Main,Weekday,18,2026-05-27,91.1,0.8,<5L,65.2,68.1,59.0,72.2,1,Need more practice sheets for weak topics,42.0
1,1405,Patna,2,NEET,Weekday,16,2024-07-08,91.9,2.4,NaN,64.9,82.0,88.4,100.0,3,Would like more one-on-one mentoring,62.0
2,1510,Mumbai,1,JEE Main,Weekday,18,2025-03-17,67.1,6.7,5-10L,90.1,86.5,91.1,76.4,5,"Great teaching pace, doubts cleared quickly",56.4
3,1456,Delhi,1,NEET,Weekday,18,2025-06-16,70.7,2.0,10-20L,29.7,19.4,24.9,NaN,3,Need more practice sheets for weak topics,31.7
4,1202,Patna,2,JEE Advanced,Weekday,17,2026-04-16,69.3,6.5,>20L,70.0,63.8,74.5,69.1,4,Need more practice sheets for weak topics,44.5


In [5]:
missing_summary = df.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
missing_summary.to_frame("missing_count").assign(
    missing_pct=lambda d: (d["missing_count"] / len(df) * 100).round(2)
)


,missing_count,missing_pct
feedback_text,68,11.33
weekly_study_hours,36,6.00
income_bracket,30,5.00
prev_exam_score,25,4.17
mock_test_3,21,3.50


The five columns flagged in the assignment brief all show missing values, matching the
MCAR (`weekly_study_hours`), MAR (`prev_exam_score`), MNAR (`mock_test_3`), categorical
(`income_bracket`), and text (`feedback_text`) mechanisms from Session 4.

### 3.2 Train/test split (80/20)

In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")


Train shape: (480, 17)
Test shape:  (120, 17)


### 3.3 Fit on TRAIN only, then transform both splits

We drop identifier / raw-text columns that shouldn't be numerically imputed as-is
(`student_id`, `enrollment_date`) from this demonstration and focus on the feature columns,
consistent with how Session 4 treated the dataset. `feedback_text` is included as a
categorical/text column so its missing entries are filled with the most frequent response.

In [7]:
feature_cols = [c for c in df.columns if c not in ["student_id", "enrollment_date"]]

train_features = train_df[feature_cols]
test_features = test_df[feature_cols]

imputer = CustomImputer(numeric_strategy="median", categorical_strategy="most_frequent",
                          add_missing_indicator=True)
imputer.fit(train_features)  # fit on TRAINING data only

train_imputed = imputer.transform(train_features)
test_imputed = imputer.transform(test_features)

print("Learned fill values:")
for col in imputer.columns_with_missing_:
    print(f"  {col!r} ({imputer.column_types_[col]}): {imputer.fill_values_[col]}")


Learned fill values:
  'weekly_study_hours' (numeric): 5.0
  'income_bracket' (categorical): 5-10L
  'prev_exam_score' (numeric): 66.1
  'mock_test_3' (numeric): 71.3
  'feedback_text' (categorical): Need more practice sheets for weak topics


### 3.4 Verify no missing values remain

In [8]:
original_missing_cols = imputer.columns_with_missing_

train_remaining = train_imputed[original_missing_cols].isna().sum().sum()
test_remaining = test_imputed[original_missing_cols].isna().sum().sum()

assert train_remaining == 0, "Missing values remain in the imputed TRAIN split!"
assert test_remaining == 0, "Missing values remain in the imputed TEST split!"

print(f"Remaining missing values in train (imputed columns): {train_remaining}")
print(f"Remaining missing values in test (imputed columns):  {test_remaining}")
print("PASSED: no missing values remain in either split.")


Remaining missing values in train (imputed columns): 0
Remaining missing values in test (imputed columns):  0
PASSED: no missing values remain in either split.


In [9]:
indicator_cols = [f"{c}_was_missing" for c in original_missing_cols]
train_imputed[original_missing_cols + indicator_cols].head()


,weekly_study_hours,income_bracket,prev_exam_score,mock_test_3,feedback_text,weekly_study_hours_was_missing,income_bracket_was_missing,prev_exam_score_was_missing,mock_test_3_was_missing,feedback_text_was_missing
145,2.0,5-10L,56.5,54.5,Would like more one-on-one mentoring,0,0,0,0,0
9,5.0,10-20L,57.8,34.9,Need more practice sheets for weak topics,1,0,0,0,0
375,6.2,5-10L,45.1,65.9,Need more practice sheets for weak topics,0,0,0,0,0
523,1.1,5-10L,54.3,74.0,Mock tests really helped identify gaps,0,0,0,0,0
188,2.8,<5L,78.8,87.9,"Great teaching pace, doubts cleared quickly",0,0,0,0,0


### 3.5 Mean / standard deviation before vs. after imputation (numeric columns)

In [10]:
numeric_missing_cols = [c for c in original_missing_cols
                         if imputer.column_types_[c] == "numeric"]

rows = []
for col in numeric_missing_cols:
    before = train_features[col]
    after = train_imputed[col]
    rows.append({
        "column": col,
        "mean_before": round(before.mean(), 3),
        "mean_after": round(after.mean(), 3),
        "std_before": round(before.std(), 3),
        "std_after": round(after.std(), 3),
    })

summary_stats = pd.DataFrame(rows).set_index("column")
summary_stats


,mean_before,mean_after,std_before,std_after
column,,,,
weekly_study_hours,6.064,6.009,4.350,4.242
prev_exam_score,65.825,65.838,14.371,14.022
mock_test_3,70.702,70.720,19.077,18.777


**Comment:** Because the imputer fills with the median (or mean) of the observed values,
the mean of each column barely moves. The **standard deviation shrinks** after imputation,
however, since every filled-in value sits exactly at the central statistic and contributes
zero deviation from it — this is the classic variance-deflation effect of mean/median
imputation. The effect is smallest for `weekly_study_hours` (MCAR — the missingness carries
no information) and most visible for `mock_test_3` (MNAR — the missing rows tend to cluster
among genuinely low scorers, so filling them with the median artificially raises those
students' apparent standing).

### 3.6 Sanity check against scikit-learn's `SimpleImputer`

In [11]:
all_match = True
for col in numeric_missing_cols:
    strategy = imputer.numeric_strategy
    sk_imputer = SimpleImputer(strategy=strategy)
    sk_imputer.fit(train_features[[col]])
    sk_fill_value = sk_imputer.statistics_[0]

    custom_fill_value = imputer.fill_values_[col]
    match = np.isclose(sk_fill_value, custom_fill_value)
    all_match &= match
    print(f"{col:20s} strategy={strategy:6s} "
          f"CustomImputer={custom_fill_value:.4f}  SimpleImputer={sk_fill_value:.4f}  match={match}")

# Categorical sanity check
cat_col = "income_bracket"
sk_cat_imputer = SimpleImputer(strategy="most_frequent")
sk_cat_imputer.fit(train_features[[cat_col]])
sk_cat_fill = sk_cat_imputer.statistics_[0]
custom_cat_fill = imputer.fill_values_[cat_col]
cat_match = (sk_cat_fill == custom_cat_fill)
all_match &= cat_match
print(f"{cat_col:20s} strategy=most_frequent "
      f"CustomImputer={custom_cat_fill!r}  SimpleImputer={sk_cat_fill!r}  match={cat_match}")

assert all_match, "CustomImputer fill values do not match SimpleImputer!"
print("\nPASSED: CustomImputer fill values match scikit-learn's SimpleImputer.")


weekly_study_hours   strategy=median CustomImputer=5.0000  SimpleImputer=5.0000  match=True
prev_exam_score      strategy=median CustomImputer=66.1000  SimpleImputer=66.1000  match=True
mock_test_3          strategy=median CustomImputer=71.3000  SimpleImputer=71.3000  match=True
income_bracket       strategy=most_frequent CustomImputer='5-10L'  SimpleImputer='5-10L'  match=True

PASSED: CustomImputer fill values match scikit-learn's SimpleImputer.


## 4. Section 7 (Bonus) — Per-column strategy overrides

Demonstrating `column_overrides` on two columns with different strategies:
`weekly_study_hours` forced to `'mean'` (overriding the dataset-wide `'median'` default) and
`income_bracket` explicitly set to `'most_frequent'`.

In [12]:
bonus_imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
    column_overrides={
        "weekly_study_hours": "mean",
        "income_bracket": "most_frequent",
    },
)
bonus_imputer.fit(train_features)

print("weekly_study_hours -> override='mean'")
print(f"  default-median value would have been: {train_features['weekly_study_hours'].median():.4f}")
print(f"  CustomImputer (override) fill value:   {bonus_imputer.fill_values_['weekly_study_hours']:.4f}")
print(f"  matches mean():                        "
      f"{np.isclose(bonus_imputer.fill_values_['weekly_study_hours'], train_features['weekly_study_hours'].mean())}")

print("\nincome_bracket -> override='most_frequent'")
print(f"  CustomImputer (override) fill value: {bonus_imputer.fill_values_['income_bracket']!r}")
print(f"  matches mode():                      "
      f"{bonus_imputer.fill_values_['income_bracket'] == train_features['income_bracket'].mode().iloc[0]}")


weekly_study_hours -> override='mean'
  default-median value would have been: 5.0000
  CustomImputer (override) fill value:   6.0642
  matches mean():                        True

income_bracket -> override='most_frequent'
  CustomImputer (override) fill value: '5-10L'
  matches mode():                      True


This confirms the override correctly changes the strategy used for a specific column while every other column continues to use the dataset-wide default.

## 5. Section 4.3 — Reflection Questions

**1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?**

Fitting on the training split only is what keeps the model's evaluation honest. If we fit on the
full dataset (or on the test split), the mean/median/mode used to fill missing training rows would
already contain information from the test set — a form of data leakage. The test set is supposed
to simulate genuinely unseen future data, and in a real deployment we would never have access to
its values when preparing our training pipeline. Leaking test statistics into training makes the
model look better during evaluation than it will actually perform in production, because the
"unseen" data quietly influenced the preprocessing step.

**2. Does mean/median imputation genuinely solve the problem for MNAR columns like `mock_test_3`? What does `add_missing_indicator` contribute?**

No — mean/median imputation does not solve the underlying problem for `mock_test_3`, because its
missingness is MNAR: rows are missing precisely *because* the true score was low (e.g. students who
scored very poorly may not have submitted or been recorded). Filling those rows with the median
overwrites a systematically low value with a "typical" value, which biases the column upward and
hides the very pattern that made the data missing in the first place — no amount of clever
averaging can recover the missing values' true (low) distribution. What `add_missing_indicator`
contributes is a separate signal: the `mock_test_3_was_missing` column preserves the fact that
"this row's value was unobserved and something about that is informative," letting a downstream
model learn from *the fact of the missingness itself* rather than losing that information once the
column is smoothed over with an imputed number.

**3. What happens with a brand-new column that is entirely missing in training but present in test? What should a production system do instead?**

With the current implementation, a column that is entirely missing during `fit()` would produce a
fill value from `series.mean()`/`series.median()`/`.mode()` on an all-NaN column — for numeric
columns this returns `NaN` itself (since `pandas` mean/median of an all-missing series is `NaN`),
so every corresponding value in `transform()` would remain unfilled, silently defeating the purpose
of the imputer. For categorical columns, our implementation only guards against this by falling
back to the string `"missing"`, but numerically this gap would go unnoticed until it broke a
downstream model with `NaN` inputs. A production-grade version should instead: (a) detect
all-missing columns during `fit()` and raise a clear warning or error rather than silently storing
`NaN`; (b) support an explicit fallback value (e.g. a global constant, `0`, or a `"missing"`
category) to use when no training signal is available at all; and (c) validate at `transform()`
time that the incoming columns match `feature_names_in_`, raising an informative error for any
column present in test but never seen during `fit()`, since imputing a totally unseen column is
not really "filling in missing values" so much as fabricating data with no statistical basis.
